# Secret Spot: Surf Forecasting via Graph-Based Wave Refraction

Prototype implementation for Part 3.


## Imports and constants


In [91]:
import heapq
import math

import numpy as np
import matplotlib.pyplot as plt

import networkx as nx


In [92]:
G = 9.81  # gravitational acceleration, m/s^2

INFINITY = math.inf          # travel time for cells no wave path reaches

# Neighbor steps for build_graph: (row_step, col_step)
EIGHT_NEIGHBOR_STEPS = [(-1, 0), (1, 0), (0, -1), (0, 1),       # straight
                        (-1, -1), (-1, 1), (1, -1), (1, 1)]     # diagonal
KNIGHT_STEPS = [(-1, -2), (-1, 2), (1, -2), (1, 2),             # extra 8 for 16 neighbors
                (-2, -1), (-2, 1), (2, -1), (2, 1)]

# Directions closer than this to zero count as zero (avoids floating-point noise)
DIRECTION_TOLERANCE = 1e-9


## Shared conventions

```
GRID:        depth_grid[row][col], depths in meters
             water = positive depth; land = 0 or negative
             row 0 = north edge, col 0 = west edge
CELL:        a (row, col) pair
CELL SIZE:   cell_size = (dy, dx) in meters (north-south, east-west)
DIRECTIONS:  degrees clockwise from north; swell direction = where the swell comes FROM
             (same convention buoys and Surfline use)
```


## Physics reference

Each equation is shown as it's written in physics, followed by how it appears in the code.

| Symbol | Meaning | Code name | Units |
|---|---|---|---|
| $g$ | gravitational acceleration | `G` | m/s² |
| $T$ | wave period | `period` | s |
| $h$ | water depth | `depth` | m |
| $\omega$ | angular frequency | `angular_frequency` | rad/s |
| $k$ | wavenumber (2π ÷ wavelength) | `wavenumber` | rad/m |
| $c$ | wave speed | `speed` | m/s |

### 1. Angular frequency (`wave_speed`)

$$\omega = \frac{2\pi}{T}$$

```python
angular_frequency = 2 * math.pi / period
```

### 2. Dispersion relation: the core wave equation (`wave_speed`)

Links a wave's period to its wavenumber at a given depth. It can't be solved for $k$ directly, so the code solves it numerically.

$$\omega^2 = g\,k\,\tanh(k\,h)$$

```python
residual = G * wavenumber * tanh_term - angular_frequency**2   # zero when the equation holds
```

### 3. Wave speed (`wave_speed`)

$$c = \frac{\omega}{k}$$

```python
return angular_frequency / wavenumber
```

### 4. Newton's method for solving the dispersion relation (`wave_speed`)

Rewrite the equation as $f(k) = 0$ and improve the guess repeatedly:

$$f(k) = g\,k\,\tanh(kh) - \omega^2 \qquad f'(k) = g\,\tanh(kh) + g\,k\,h\,\left(1 - \tanh^2(kh)\right)$$

$$k_{\text{new}} = k - \frac{f(k)}{f'(k)}$$

```python
residual = G * wavenumber * tanh_term - angular_frequency**2
residual_slope = G * tanh_term + G * wavenumber * depth * (1 - tanh_term**2)
new_wavenumber = wavenumber - residual / residual_slope
```

Starting guess, a common approximation that's close at any depth:

$$k_0 = \frac{\omega^2}{g} \qquad k \approx \frac{k_0}{\sqrt{\tanh(k_0 h)}}$$

```python
deep_wavenumber = angular_frequency**2 / G
wavenumber = deep_wavenumber / math.sqrt(math.tanh(deep_wavenumber * depth))
```

### 5. Limits used as tests (`wave_speed` tests)

Deep water ($\tanh(kh) \to 1$):
$$c_{\text{deep}} = \frac{g\,T}{2\pi}$$
```python
deep_limit = G * test_period / (2 * math.pi)
```

Shallow water ($\tanh(kh) \to kh$):
$$c_{\text{shallow}} = \sqrt{g\,h}$$
```python
shallow_limit = math.sqrt(G * 0.5)
```

### 6. Travel time across one edge (`build_graph`)

Distance between two cell centers, with cell size $(\Delta y, \Delta x)$ and steps of $\Delta r$ rows and $\Delta c$ columns:

$$\Delta s = \sqrt{(\Delta r\,\Delta y)^2 + (\Delta c\,\Delta x)^2}$$

Travel time, averaging the "slowness" ($1/c$) of the two cells:

$$\Delta t = \frac{\Delta s}{2}\left(\frac{1}{c_1} + \frac{1}{c_2}\right)$$

```python
distance = math.sqrt((row_step * dy)**2 + (col_step * dx)**2)
travel_time = (distance / 2) * (1 / speed_1 + 1 / speed_2)
```

### 7. Converting degrees to meters (`load_bathymetry`)

$$\Delta y = \Delta\text{lat} \times 111{,}000 \qquad \Delta x = \Delta\text{lon} \times 111{,}000 \times \cos(\text{latitude})$$

```python
dy = lat_spacing * METERS_PER_DEGREE
dx = lon_spacing * METERS_PER_DEGREE * math.cos(math.radians(mean_latitude))
```

### 8. Staggered start times along the wave front (`get_source_cells`)

The swell travels toward bearing $\beta$ = swell direction + 180°. A cell's position along that direction, and its start time with reference speed $c_{\text{ref}}$:

$$p = y_{\text{north}}\cos\beta + x_{\text{east}}\sin\beta \qquad t_{\text{start}} = \frac{p - p_{\min}}{c_{\text{ref}}}$$

```python
position_along_travel = north_position * travel_north + east_position * travel_east
start_time = (position_along_travel - earliest) / reference_speed
```


# 1. Grid building


In [93]:
def make_synthetic_grid(shape, rows, cols, base_depth, cell_size):
    """
    Build a synthetic seafloor for testing.

    Inputs:
        shape (str): "flat", "mound", or "slope"
        rows, cols (int): grid size, at least 1 each ("slope" needs at least 2 columns)
        base_depth (float): depth of open water in meters (positive)
        cell_size (tuple): (dy, dx) in meters; not used to build the grid, but kept
                           so every grid-building function shares the same inputs
    Returns:
        depth_grid (np.ndarray): rows x cols array of depths in meters
                                 (water > 0, land <= 0)
    Assumptions:
        Row 0 is the north edge and column 0 is the west edge.
        "mound" is centered in the grid; "slope" has its beach (depth 0) on the east edge.
    """
    if rows < 1 or cols < 1:
        raise ValueError("grid must have at least one row and column")

    depth_grid = np.full((rows, cols), float(base_depth))

    if shape == "flat":
        return depth_grid

    elif shape == "mound":
        center_row, center_col = rows // 2, cols // 2
        radius = min(rows, cols) // 4
        peak_depth = base_depth / 4
        for r in range(rows):
            for c in range(cols):
                distance_from_center = math.hypot(r - center_row, c - center_col)
                if distance_from_center < radius:
                    # shallowest at the center, back to base_depth at the edge
                    depth_grid[r, c] = peak_depth + (base_depth - peak_depth) * (distance_from_center / radius)
        return depth_grid

    elif shape == "slope":
        if cols < 2:
            raise ValueError("slope needs at least 2 columns")
        for c in range(cols):
            depth_grid[:, c] = base_depth * (1 - c / (cols - 1))
        return depth_grid  # last column is 0 = land (the beach)

    else:
        raise ValueError(f"unknown shape: {shape!r}")

In [94]:
# Test: make_synthetic_grid
flat = make_synthetic_grid("flat", 4, 5, 20.0, (10, 10))
assert flat.shape == (4, 5), "flat grid has wrong shape"
assert np.all(flat == 20.0), "flat grid should be 20 m everywhere"

mound = make_synthetic_grid("mound", 20, 20, 20.0, (10, 10))
assert mound[10, 10] == 5.0, "mound center should be base_depth / 4"
assert mound[0, 0] == 20.0, "corner should be untouched base_depth"
assert mound.min() == 5.0 and mound.max() == 20.0, "mound depths out of range"

slope = make_synthetic_grid("slope", 3, 5, 20.0, (10, 10))
assert np.all(slope[:, 0] == 20.0), "west edge should be base_depth"
assert np.all(slope[:, -1] == 0.0), "east edge should be 0 (beach)"
assert np.all(np.diff(slope[0]) < 0), "depth should decrease toward the east"

# invalid inputs should raise errors
for bad_call in [lambda: make_synthetic_grid("flat", 0, 5, 20, (10, 10)),
                 lambda: make_synthetic_grid("reef", 5, 5, 20, (10, 10)),
                 lambda: make_synthetic_grid("slope", 5, 1, 20, (10, 10))]:
    try:
        bad_call()
        raise AssertionError("expected a ValueError")
    except ValueError:
        pass

print("make_synthetic_grid: all tests passed")

make_synthetic_grid: all tests passed


In [95]:
def load_bathymetry(file_path, bounds):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: file_path (string), bounds = (south, north, west, east) in degrees
    # Returns: depth_grid (2D array), cell_size (dy, dx) in meters
    #
    # read elevation values, latitudes, and longitudes from file
    # keep only points inside bounds
    #
    # if no points remain:
    #     raise error "bounds contain no data"
    #
    # GEBCO and NOAA store ELEVATION: negative below sea level
    # depth_grid = -elevation                    # flip sign so water depth is positive
    #
    # for each cell with a missing value:
    #     set depth to 0                         # treat as land, so no wave path crosses it
    #     record a warning with the count of missing cells
    #
    # convert grid spacing from degrees to meters
    # dy = latitude spacing  * 111,000
    # dx = longitude spacing * 111,000 * cos(average latitude)
    #
    # return depth_grid, (dy, dx)
    raise NotImplementedError


In [96]:
# Test: load_bathymetry
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [97]:
def wave_speed(depth, period):
    """
    Speed of a wave of a given period at a given depth.

    Inputs:
        depth (float): water depth in meters (land if <= 0)
        period (float): wave period in seconds (must be > 0)
    Returns:
        speed (float) in m/s, or None for land
    Assumptions:
        Uses the full wave speed (dispersion) equation,
        angular_frequency^2 = g * wavenumber * tanh(wavenumber * depth),
        solved for the wavenumber with Newton's method, so it works in deep,
        intermediate, and shallow water.
    """
    if period <= 0:
        raise ValueError("period must be positive")
    if depth <= 0:
        return None

    angular_frequency = 2 * math.pi / period
    deep_wavenumber = angular_frequency**2 / G
    # good starting guess at any depth
    wavenumber = deep_wavenumber / math.sqrt(math.tanh(deep_wavenumber * depth))

    for _ in range(50):
        tanh_term = math.tanh(wavenumber * depth)
        residual = G * wavenumber * tanh_term - angular_frequency**2        # how far off the guess is
        residual_slope = G * tanh_term + G * wavenumber * depth * (1 - tanh_term**2)
        new_wavenumber = wavenumber - residual / residual_slope              # Newton step
        if abs(new_wavenumber - wavenumber) < 1e-12 * wavenumber:
            wavenumber = new_wavenumber
            break
        wavenumber = new_wavenumber
    else:
        raise RuntimeError("wave_speed did not converge")

    return angular_frequency / wavenumber   # speed = angular frequency / wavenumber

In [98]:
# Test: wave_speed
test_period = 10.0

# deep water should match g * period / (2 * pi)
deep = wave_speed(1000, test_period)
deep_limit = G * test_period / (2 * math.pi)
assert abs(deep - deep_limit) / deep_limit < 0.001, f"deep: got {deep:.3f}, expected {deep_limit:.3f}"

# very shallow water should match sqrt(g * depth)
shallow = wave_speed(0.5, test_period)
shallow_limit = math.sqrt(G * 0.5)
assert abs(shallow - shallow_limit) / shallow_limit < 0.01, f"shallow: got {shallow:.3f}, expected {shallow_limit:.3f}"

# speed should increase with depth
speeds = [wave_speed(test_depth, test_period) for test_depth in [1, 5, 10, 20, 50, 100]]
assert all(slower < faster for slower, faster in zip(speeds, speeds[1:])), "speed should increase with depth"

# the answer should satisfy the wave equation at many depths and periods
for test_depth in [0.01, 0.1, 1, 5, 20, 49, 200, 5000]:
    for period in [2, 6, 10, 15, 25]:
        angular_frequency = 2 * math.pi / period
        wavenumber = angular_frequency / wave_speed(test_depth, period)
        residual = G * wavenumber * math.tanh(wavenumber * test_depth) - angular_frequency**2
        assert abs(residual) < 1e-9, f"failed at depth {test_depth}, period {period}"

# land and bad input
assert wave_speed(0, test_period) is None and wave_speed(-5, test_period) is None, "land should return None"
try:
    wave_speed(10, 0)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass

print("wave_speed: all tests passed")
print(f"  10 s swell: 1000 m -> {deep:.2f} m/s, 49 m -> {wave_speed(49, test_period):.2f} m/s, 0.5 m -> {shallow:.2f} m/s")

wave_speed: all tests passed
  10 s swell: 1000 m -> 15.61 m/s, 49 m -> 15.09 m/s, 0.5 m -> 2.21 m/s


In [99]:
def passes_land(speed_grid, row, col, row_step, col_step):
    """
    Check whether a move between two water cells cuts through or past land.

    Inputs:
        speed_grid (list of lists): wave speed per cell, None for land
        row, col (int): starting cell
        row_step, col_step (int): the move (one of the 8- or 16-neighbor steps)
    Returns:
        True if any cell the move passes between is land, otherwise False
    Assumptions:
        Straight moves pass no in-between cells. A diagonal passes between the two
        cells that share its corner. A knight move (e.g., 1 down, 2 right) passes
        between the two cells along its middle.
    """
    if abs(row_step) + abs(col_step) == 1:                     # straight move
        in_between = []
    elif abs(row_step) == 1 and abs(col_step) == 1:            # diagonal
        in_between = [(row + row_step, col), (row, col + col_step)]
    elif abs(row_step) == 1:                                   # knight move, mostly sideways
        half_col = col_step // 2
        in_between = [(row, col + half_col), (row + row_step, col + half_col)]
    else:                                                      # knight move, mostly up/down
        half_row = row_step // 2
        in_between = [(row + half_row, col), (row + half_row, col + col_step)]

    return any(speed_grid[between_row][between_col] is None
               for between_row, between_col in in_between)

In [100]:
# Test: passes_land
# 3 x 3 water grid with one land cell in the middle of the top row, at (0, 1)
speeds = [[1.0, None, 1.0],
          [1.0, 1.0,  1.0],
          [1.0, 1.0,  1.0]]
assert passes_land(speeds, 1, 1, 0, 1) is False, "straight moves never pass land"
assert passes_land(speeds, 0, 0, 1, 1) is True, "diagonal (0,0)->(1,1) cuts the corner of (0,1)"
assert passes_land(speeds, 1, 0, 1, 1) is False, "diagonal (1,0)->(2,1) is all water"
assert passes_land(speeds, 1, 0, -1, 2) is True, "knight (1,0)->(0,2) passes over (0,1)"
assert passes_land(speeds, 2, 0, -1, 2) is False, "knight (2,0)->(1,2) stays in water"

# 3 x 2 grid with land at (1, 1), for an up/down knight move
tall_speeds = [[1.0, 1.0],
               [1.0, None],
               [1.0, 1.0]]
assert passes_land(tall_speeds, 2, 0, -2, 1) is True, "knight (2,0)->(0,1) passes over (1,1)"

print("passes_land: all tests passed")

passes_land: all tests passed


In [101]:
def build_graph(depth_grid, cell_size, period, neighbors):
    """
    Turn a depth grid into a weighted graph of wave travel times.

    Inputs:
        depth_grid (np.ndarray): depths in meters (water > 0, land <= 0)
        cell_size (tuple): (dy, dx) in meters
        period (float): wave period in seconds
        neighbors (int): 8 or 16 connections per cell
    Returns:
        graph (dict): (row, col) -> list of ((neighbor_row, neighbor_col), travel_time)
                      Every water cell is a key, even if it has no edges; land has no node.
    Assumptions:
        Travel time between two cells uses the average "slowness" (1 / speed) of both,
        so a slow, shallow cell counts fully.
    """
    if neighbors == 8:
        steps = EIGHT_NEIGHBOR_STEPS
    elif neighbors == 16:
        steps = EIGHT_NEIGHBOR_STEPS + KNIGHT_STEPS
    else:
        raise ValueError("neighbors must be 8 or 16")

    dy, dx = cell_size
    rows, cols = depth_grid.shape

    # compute each cell's speed once, not once per edge
    speed_grid = [[wave_speed(depth_grid[r, c], period) for c in range(cols)]
                  for r in range(rows)]

    graph = {}
    for r in range(rows):
        for c in range(cols):
            if speed_grid[r][c] is None:
                continue                                       # land: no node
            graph[(r, c)] = []

            for row_step, col_step in steps:
                neighbor_row, neighbor_col = r + row_step, c + col_step
                if not (0 <= neighbor_row < rows and 0 <= neighbor_col < cols):
                    continue                                   # outside the grid
                if speed_grid[neighbor_row][neighbor_col] is None:
                    continue                                   # neighbor is land
                if passes_land(speed_grid, r, c, row_step, col_step):
                    continue                                   # EDGE CASE: no shortcuts through land

                distance = math.hypot(row_step * dy, col_step * dx)
                travel_time = (distance / 2) * (1 / speed_grid[r][c]
                                                + 1 / speed_grid[neighbor_row][neighbor_col])
                graph[(r, c)].append(((neighbor_row, neighbor_col), travel_time))

    return graph

In [102]:
# Test: build_graph
test_period = 10.0
speed_20m = wave_speed(20, test_period)

# 1. Flat 3x3 grid: correct number of edges and correct times
flat_graph = build_graph(make_synthetic_grid("flat", 3, 3, 20.0, (10, 10)), (10, 10), test_period, 8)
assert len(flat_graph) == 9, "every water cell should be a node"
assert len(flat_graph[(1, 1)]) == 8, "center should have 8 neighbors"
assert len(flat_graph[(0, 0)]) == 3, "corner should have 3 neighbors"
edge_times = dict(flat_graph[(1, 1)])
assert math.isclose(edge_times[(1, 2)], 10 / speed_20m), "straight edge time is wrong"
assert math.isclose(edge_times[(2, 2)], math.hypot(10, 10) / speed_20m), "diagonal edge time is wrong"

# 2. Non-square cells: east-west steps use dx, north-south steps use dy
wide_graph = build_graph(make_synthetic_grid("flat", 3, 3, 20.0, (10, 5)), (10, 5), test_period, 8)
edge_times = dict(wide_graph[(1, 1)])
assert math.isclose(edge_times[(1, 2)], 5 / speed_20m), "east step should use dx"
assert math.isclose(edge_times[(0, 1)], 10 / speed_20m), "north step should use dy"

# 3. Land cells get no node, and no edges lead to them
island_grid = make_synthetic_grid("flat", 3, 3, 20.0, (10, 10))
island_grid[1, 1] = 0.0
island_graph = build_graph(island_grid, (10, 10), test_period, 8)
assert (1, 1) not in island_graph, "land should not be a node"
assert all(neighbor != (1, 1) for edges in island_graph.values() for neighbor, _ in edges), "edge into land"

# 4. A diagonal can't cut the corner of a land cell
corner_grid = make_synthetic_grid("flat", 2, 2, 20.0, (10, 10))
corner_grid[0, 1] = 0.0
corner_graph = build_graph(corner_grid, (10, 10), test_period, 8)
assert (1, 1) not in dict(corner_graph[(0, 0)]), "diagonal cut past land"

# 5. 16 neighbors: knight moves added, but never over land
knight_graph = build_graph(make_synthetic_grid("flat", 5, 5, 20.0, (10, 10)), (10, 10), test_period, 16)
assert len(knight_graph[(2, 2)]) == 16, "center should have 16 neighbors"
assert math.isclose(dict(knight_graph[(2, 2)])[(3, 4)], math.hypot(10, 20) / speed_20m), "knight time is wrong"
knight_land_grid = make_synthetic_grid("flat", 5, 5, 20.0, (10, 10))
knight_land_grid[2, 3] = 0.0
knight_land_graph = build_graph(knight_land_grid, (10, 10), test_period, 16)
assert (3, 4) not in dict(knight_land_graph[(2, 2)]), "knight move jumped over land"

# 6. Mixed depths: travel time averages the two cells' slowness
step_grid = np.array([[20.0, 5.0]])
step_graph = build_graph(step_grid, (10, 10), test_period, 8)
expected_time = (10 / 2) * (1 / speed_20m + 1 / wave_speed(5, test_period))
assert math.isclose(dict(step_graph[(0, 0)])[(0, 1)], expected_time), "mixed-depth time is wrong"

# 7. Every edge goes both ways with the same time
mound_graph = build_graph(make_synthetic_grid("mound", 12, 12, 20.0, (10, 10)), (10, 10), test_period, 16)
for cell, edges in mound_graph.items():
    for neighbor, edge_time in edges:
        assert math.isclose(dict(mound_graph[neighbor])[cell], edge_time), f"asymmetric edge {cell}-{neighbor}"

# 8. Works with dijkstra: on a flat grid, the diagonal route is shortest
travel_times, _ = dijkstra(flat_graph, {(0, 0): 0})
assert math.isclose(travel_times[(2, 2)], 2 * math.hypot(10, 10) / speed_20m), "flat-grid route is wrong"

# 9. Invalid neighbor count
try:
    build_graph(make_synthetic_grid("flat", 3, 3, 20.0, (10, 10)), (10, 10), test_period, 4)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass

print("build_graph: all tests passed")
print(f"  12 x 12 mound grid, 16 neighbors: {len(mound_graph)} nodes, "
      f"{sum(len(edges) for edges in mound_graph.values())} edges")

# Expected Output
# passes_land: all tests passed
# build_graph: all tests passed
# 12 x 12 mound grid, 16 neighbors: 144 nodes, 1892 edges

build_graph: all tests passed
  12 x 12 mound grid, 16 neighbors: 144 nodes, 1892 edges


# 2. Refraction (Dijkstra)


In [103]:
def get_source_cells(depth_grid, cell_size, period, swell_direction):
    """
    Find the edge cells where incoming swell enters the grid, with each one's start time.

    Inputs:
        depth_grid (np.ndarray): depths in meters (water > 0, land <= 0)
        cell_size (tuple): (dy, dx) in meters
        period (float): wave period in seconds
        swell_direction (float): degrees clockwise from north that the swell comes FROM
    Returns:
        sources (dict): (row, col) -> start time in seconds (earliest cell = 0);
                        empty if no water edge faces the swell
    Assumptions:
        The swell is one straight wave front. Start times use a single reference speed
        (the average speed over the entry cells), which assumes the offshore edge has
        fairly uniform depth.
    """
    dy, dx = cell_size
    rows, cols = depth_grid.shape

    travel_bearing = math.radians((swell_direction + 180) % 360)   # swell travels AWAY from its source
    travel_north = math.cos(travel_bearing)
    travel_east = math.sin(travel_bearing)

    # edge cells where the swell enters the grid
    candidates = set()
    if travel_north < -DIRECTION_TOLERANCE:                      # heading south: enters at the north edge
        candidates.update((0, c) for c in range(cols))
    if travel_north > DIRECTION_TOLERANCE:                       # heading north: enters at the south edge
        candidates.update((rows - 1, c) for c in range(cols))
    if travel_east > DIRECTION_TOLERANCE:                        # heading east: enters at the west edge
        candidates.update((r, 0) for r in range(rows))
    if travel_east < -DIRECTION_TOLERANCE:                       # heading west: enters at the east edge
        candidates.update((r, cols - 1) for r in range(rows))
    candidates = {cell for cell in candidates if depth_grid[cell] > 0}   # drop land

    if not candidates:
        return {}                                                # EDGE CASE: swell comes from the land side

    # stagger start times along the wave front
    reference_speed = sum(wave_speed(depth_grid[cell], period) for cell in candidates) / len(candidates)
    position_along_travel = {}
    for row, col in candidates:
        north_position = -row * dy                               # rows increase southward
        east_position = col * dx
        position_along_travel[(row, col)] = north_position * travel_north + east_position * travel_east

    earliest = min(position_along_travel.values())
    sources = {cell: (position - earliest) / reference_speed
               for cell, position in position_along_travel.items()}
    return sources

In [104]:
# Test: get_source_cells
test_period = 10.0
flat_grid = make_synthetic_grid("flat", 4, 5, 20.0, (10, 10))
speed_20m = wave_speed(20, test_period)

# 1. Swell from the west enters along the west edge only, all at once
from_west = get_source_cells(flat_grid, (10, 10), test_period, 270)
assert set(from_west) == {(r, 0) for r in range(4)}, f"wrong entry cells: {sorted(from_west)}"
assert all(math.isclose(start, 0, abs_tol=1e-9) for start in from_west.values()), "straight-on swell should start together"

# 2. Swell from the north enters along the north edge only
from_north = get_source_cells(flat_grid, (10, 10), test_period, 0)
assert set(from_north) == {(0, c) for c in range(5)}, f"wrong entry cells: {sorted(from_north)}"

# 3. Swell from the northwest enters along two edges, staggered in time
from_northwest = get_source_cells(flat_grid, (10, 10), test_period, 315)
expected_cells = {(0, c) for c in range(5)} | {(r, 0) for r in range(4)}
assert set(from_northwest) == expected_cells, "should enter along the north and west edges"
assert from_northwest[(0, 0)] == 0, "the northwest corner is reached first"
assert math.isclose(from_northwest[(0, 1)], from_northwest[(1, 0)]), "cells on the same wave front start together"
expected_delay = 2 * 10 * math.sin(math.radians(45)) / speed_20m
assert math.isclose(from_northwest[(0, 2)], expected_delay), "start-time delay is wrong"

# 4. Swell from the land side gives no sources
slope_grid = make_synthetic_grid("slope", 4, 5, 20.0, (10, 10))   # east edge is land (the beach)
assert get_source_cells(slope_grid, (10, 10), test_period, 90) == {}, "swell from land should give no sources"

# 5. Land cells on the entry edge are skipped
rocky_grid = make_synthetic_grid("flat", 4, 5, 20.0, (10, 10))
rocky_grid[2, 0] = 0.0
from_west_rocky = get_source_cells(rocky_grid, (10, 10), test_period, 270)
assert (2, 0) not in from_west_rocky and len(from_west_rocky) == 3, "land on the edge should be skipped"

# 6. Non-square cells: east-west spacing (dx) sets the delay along the north edge
wide_grid = make_synthetic_grid("flat", 4, 5, 20.0, (10, 5))
from_northwest_wide = get_source_cells(wide_grid, (10, 5), test_period, 315)
assert math.isclose(from_northwest_wide[(0, 2)], 2 * 5 * math.sin(math.radians(45)) / speed_20m), "should use dx"

# 7. Start times are never negative, and 360 degrees equals 0 degrees
for direction in range(0, 360, 10):
    assert min(get_source_cells(flat_grid, (10, 10), test_period, direction).values()) == 0, f"bad start at {direction}"
assert get_source_cells(flat_grid, (10, 10), test_period, 360) == from_north, "360 should equal 0"

print("get_source_cells: all tests passed")
print(f"  swell from NW on a 4 x 5 grid: {len(from_northwest)} entry cells, "
      f"latest start {max(from_northwest.values()):.2f} s")

# Expected Output
# get_source_cells: all tests passed
# swell from NW on a 4 x 5 grid: 8 entry cells, latest start 2.33 s


get_source_cells: all tests passed
  swell from NW on a 4 x 5 grid: 8 entry cells, latest start 2.33 s


In [105]:
def dijkstra(graph, sources):
    """
    Multi-source Dijkstra's algorithm: fastest arrival time at every node.

    Inputs:
        graph (dict): node -> list of (neighbor, edge_time); edge times must be >= 0
        sources (dict): starting node -> start time (seconds)
    Returns:
        travel_times (dict): node -> fastest arrival time (INFINITY if never reached)
        predecessors (dict): node -> previous node on the fastest path
                             (None for starting nodes and unreached nodes)
    Assumptions:
        Every node appears as a key in graph, even if it has no edges.
        Starting nodes that are not in graph (e.g., land cells) are skipped.
    """
    # A. Initialization 
    travel_times = {node: INFINITY for node in graph}
    predecessors = {node: None for node in graph}
    finalized = set()
    queue = []                                   # heap of (time, node); smallest time first

    # B. Load the starting nodes
    for node, start_time in sources.items():
        if node not in graph:
            continue                             # skip sources that aren't in the graph
        if start_time < travel_times[node]:
            travel_times[node] = start_time
            heapq.heappush(queue, (start_time, node))

    # C. Main loop
    while queue:
        current_time, current = heapq.heappop(queue)        # C1. extract-min
        if current in finalized:
            continue                                         # C2. skip outdated entries
        finalized.add(current)                               # C3. finalize

        for neighbor, edge_time in graph[current]:           # C4. relaxation
            if neighbor in finalized:
                continue
            new_time = current_time + edge_time
            if new_time < travel_times[neighbor]:
                travel_times[neighbor] = new_time
                predecessors[neighbor] = current
                heapq.heappush(queue, (new_time, neighbor))

    # D. Termination 
    # queue is empty: every reachable node is finalized;
    # nodes still at INFINITY were never reached
    return travel_times, predecessors

In [106]:
# Test: dijkstra

def make_undirected_graph(edge_list):
    """Test helper: build a graph dict from (node_a, node_b, weight) edges, both directions."""
    graph = {}
    for node_a, node_b, weight in edge_list:
        graph.setdefault(node_a, []).append((node_b, weight))
        graph.setdefault(node_b, []).append((node_a, weight))
    return graph

# 6-node example from Javaid, "Understanding Dijkstra's Algorithm"
ieee_edges = [("A", "B", 2), ("A", "C", 5), ("A", "D", 1), ("B", "C", 3), ("B", "D", 2),
              ("C", "D", 3), ("C", "E", 1), ("C", "F", 5), ("D", "E", 1), ("E", "F", 2)]
ieee_graph = make_undirected_graph(ieee_edges)

# 1. Single source: should match the paper's worked answer
travel_times, predecessors = dijkstra(ieee_graph, {"A": 0})
expected_times = {"A": 0, "B": 2, "C": 3, "D": 1, "E": 2, "F": 4}
expected_predecessors = {"A": None, "B": "A", "C": "E", "D": "A", "E": "D", "F": "E"}
assert travel_times == expected_times, f"times differ from paper: {travel_times}"
assert predecessors == expected_predecessors, f"predecessors differ from paper: {predecessors}"

# 2. Should match NetworkX's built-in Dijkstra
nx_graph = nx.Graph()
nx_graph.add_weighted_edges_from(ieee_edges)
assert travel_times == nx.single_source_dijkstra_path_length(nx_graph, "A"), "differs from NetworkX"

# 3. Multiple sources with different start times
travel_times, predecessors = dijkstra(ieee_graph, {"A": 0, "F": 1})
assert travel_times["F"] == 1 and predecessors["F"] is None, "F should keep its own start time"
assert travel_times["E"] == 2 and predecessors["E"] == "D", "E is still reached fastest from A"

# 4. Unreachable node, and a source that isn't in the graph
island_graph = make_undirected_graph(ieee_edges)
island_graph["G"] = []                                 # node with no edges
travel_times, predecessors = dijkstra(island_graph, {"A": 0, "land": 0})
assert travel_times["G"] == INFINITY and predecessors["G"] is None, "G should be unreachable"
assert "land" not in travel_times, "sources outside the graph should be skipped"

# 5. No sources: nothing is reached
travel_times, _ = dijkstra(ieee_graph, {})
assert all(time == INFINITY for time in travel_times.values()), "nothing should be reached"

# 6. Invariant: after Dijkstra, no edge offers a shortcut
travel_times, _ = dijkstra(ieee_graph, {"A": 0})
for node, edges in ieee_graph.items():
    for neighbor, edge_time in edges:
        assert travel_times[neighbor] <= travel_times[node] + edge_time, f"shortcut {node} -> {neighbor}"

print("dijkstra: all tests passed")
print("  shortest times from A:", dict(sorted(travel_times.items())))

# Expected Output  
# dijkstra: all tests passed
# shortest times from A: {'A': 0, 'B': 2, 'C': 3, 'D': 1, 'E': 2, 'F': 4}

dijkstra: all tests passed
  shortest times from A: {'A': 0, 'B': 2, 'C': 3, 'D': 1, 'E': 2, 'F': 4}


In [107]:
def reconstruct_path(predecessors, travel_times, cell):
    """
    Trace the fastest path to a cell back to its offshore starting cell.

    Inputs:
        predecessors (dict): node -> previous node on the fastest path (from dijkstra)
        travel_times (dict): node -> fastest arrival time (from dijkstra)
        cell: the node to trace back from
    Returns:
        path (list): nodes from the starting node to cell; empty if cell was never reached
    Assumptions:
        Starting nodes have predecessor None, so the trace stops there.
    """
    if cell not in travel_times or travel_times[cell] == INFINITY:
        return []                                         # EDGE CASE: unreachable cell

    path = [cell]
    while predecessors[path[-1]] is not None:
        path.append(predecessors[path[-1]])
        if len(path) > len(predecessors):
            raise RuntimeError("cycle in predecessors")   # safety check: should never happen

    path.reverse()                                        # now reads start -> cell
    return path

In [108]:
# Test: reconstruct_path (reuses ieee_graph from the dijkstra test)
ieee_graph["G"] = []                                      # an unreachable node
travel_times, predecessors = dijkstra(ieee_graph, {"A": 0})

assert reconstruct_path(predecessors, travel_times, "F") == ["A", "D", "E", "F"], "path to F is wrong"
assert reconstruct_path(predecessors, travel_times, "C") == ["A", "D", "E", "C"], "path to C is wrong"
assert reconstruct_path(predecessors, travel_times, "A") == ["A"], "path to the start is just the start"
assert reconstruct_path(predecessors, travel_times, "G") == [], "unreachable node should give []"
assert reconstruct_path(predecessors, travel_times, "Z") == [], "unknown node should give []"

# travel times must increase along every path
for node in "ABCDEF":
    path = reconstruct_path(predecessors, travel_times, node)
    path_times = [travel_times[step] for step in path]
    assert all(earlier < later for earlier, later in zip(path_times, path_times[1:])), f"times not increasing to {node}"

print("reconstruct_path: all tests passed")

# Exptected Output
# reconstruct_path: all tests passed


reconstruct_path: all tests passed


In [109]:
def count_converging_paths(predecessors, travel_times, pin, radius):
    """
    Count how many distinct offshore starting cells feed swell into the area around the pin.

    Inputs:
        predecessors, travel_times (dict): output of dijkstra
        pin (tuple): (row, col) of the spot of interest
        radius (float): how far from the pin to look, in cells
    Returns:
        count (int): number of distinct starting cells whose fastest paths end near the pin;
                     0 means the spot is blocked from this swell direction
    Assumptions:
        The pin may sit on land (the beach); only water cells within the radius are used.
    """
    pin_row, pin_col = pin
    nearby_water = [cell for cell in travel_times
                    if math.hypot(cell[0] - pin_row, cell[1] - pin_col) <= radius]

    if not nearby_water:
        raise ValueError("no water near pin: move the pin or increase radius")   # EDGE CASE

    feeding_sources = set()
    for cell in nearby_water:
        path = reconstruct_path(predecessors, travel_times, cell)
        if path:
            feeding_sources.add(path[0])                  # path[0] = the offshore starting cell

    return len(feeding_sources)

In [110]:
# Test: count_converging_paths
test_period = 10.0
cell_size = (50, 50)
flat_grid = make_synthetic_grid("flat", 21, 21, 20.0, cell_size)

# 1. Flat seafloor, swell from the west: paths run straight along rows,
#    so a radius-1 pin is fed by 3 starting cells (its row and the rows above and below)
flat_graph = build_graph(flat_grid, cell_size, test_period, 8)
travel_times, predecessors = dijkstra(flat_graph, get_source_cells(flat_grid, cell_size, test_period, 270))
assert count_converging_paths(predecessors, travel_times, (10, 20), 1) == 3, "flat grid should give 3"

# 2. A wall of land blocks everything behind it
wall_grid = flat_grid.copy()
wall_grid[:, 10] = 0.0
wall_graph = build_graph(wall_grid, cell_size, test_period, 8)
travel_times, predecessors = dijkstra(wall_graph, get_source_cells(wall_grid, cell_size, test_period, 270))
assert count_converging_paths(predecessors, travel_times, (10, 20), 1) == 0, "wall should block the pin"

# 3. A pin with no water nearby raises an error
land_grid = flat_grid.copy()
land_grid[5:16, 15:21] = 0.0
land_graph = build_graph(land_grid, cell_size, test_period, 8)
travel_times, predecessors = dijkstra(land_graph, get_source_cells(land_grid, cell_size, test_period, 270))
try:
    count_converging_paths(predecessors, travel_times, (10, 19), 1)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass

print("count_converging_paths: all tests passed")

# Expected Output
# count_converging_paths: all tests passed

count_converging_paths: all tests passed


In [111]:
def swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors):
    """
    Measure how exposed a spot is to swell from each direction.

    Inputs:
        depth_grid (np.ndarray), cell_size (tuple): the seafloor
        pin (tuple): (row, col) of the spot of interest
        period (float): wave period in seconds
        directions (list): swell directions in degrees (where swell comes FROM)
        radius (float): search radius around the pin, in cells
        neighbors (int): 8 or 16
    Returns:
        profile (dict): direction -> converging path count (0 = blocked)
    Assumptions:
        The graph depends only on the seafloor and the period, so it is built once
        and reused for every direction. Every water cell on the grid's edge is treated
        as open ocean.
    """
    graph = build_graph(depth_grid, cell_size, period, neighbors)   # built ONCE

    profile = {}
    for direction in directions:
        sources = get_source_cells(depth_grid, cell_size, period, direction)
        if not sources:
            profile[direction] = 0                                  # swell from the land side
            continue
        travel_times, predecessors = dijkstra(graph, sources)
        profile[direction] = count_converging_paths(predecessors, travel_times, pin, radius)

    return profile


In [112]:
# Test: swell_window_profile
test_period = 10.0
cell_size = (50, 50)
all_directions = list(range(0, 360, 10))

# 1. Flat seafloor, pin in the center: every direction is equally open
flat_grid = make_synthetic_grid("flat", 21, 21, 20.0, cell_size)
center_profile = swell_window_profile(flat_grid, cell_size, (10, 10), test_period, all_directions, 2, 16)
assert set(center_profile) == set(all_directions), "profile should cover every direction"
assert len(set(center_profile.values())) == 1, f"flat grid should be equal in all directions: {center_profile}"

# 2. East-facing beach: swell from the land side (east) is blocked,
#    and swell from offshore (west) gives the highest exposure
slope_grid = make_synthetic_grid("slope", 21, 21, 20.0, cell_size)
beach_profile = swell_window_profile(slope_grid, cell_size, (10, 19), test_period, all_directions, 1, 16)
assert beach_profile[90] == 0, "swell from land should be blocked"
assert beach_profile[270] == max(beach_profile.values()), "offshore swell should be most exposed"

# 3. Shortest paths go AROUND a shallow mound rather than over it
#    (Dijkstra keeps the fastest arrival; see note on focusing)
mound_grid = make_synthetic_grid("mound", 41, 41, 20.0, cell_size)
mound_graph = build_graph(mound_grid, cell_size, test_period, 16)
travel_times, predecessors = dijkstra(mound_graph, get_source_cells(mound_grid, cell_size, test_period, 270))
path_behind_mound = reconstruct_path(predecessors, travel_times, (20, 32))
shallowest_on_path = min(mound_grid[cell] for cell in path_behind_mound)
assert shallowest_on_path > mound_grid.min(), "fastest path should avoid the mound's shallow top"

print("swell_window_profile: all tests passed")
print(f"  east-facing beach, swell from W: {beach_profile[270]}, from E: {beach_profile[90]}")
print(f"  path behind the mound never shallower than {shallowest_on_path:.1f} m (mound top: {mound_grid.min():.1f} m)")

# Expected Output
# swell_window_profile: all tests passed
# east-facing beach, swell from W: 3, from E: 0
# path behind the mound never shallower than 13.1 m (mound top: 5.0 m)

swell_window_profile: all tests passed
  east-facing beach, swell from W: 3, from E: 0
  path behind the mound never shallower than 13.1 m (mound top: 5.0 m)


# 3. Scoring


In [113]:
def angle_difference(a, b):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: two directions in degrees (float)
    # Returns: smallest angle between them, 0 to 180 (float)
    #
    # diff = |a - b| mod 360
    # if diff > 180:
    #     diff = 360 - diff                      # EDGE CASE: wrap around north
    # return diff
    raise NotImplementedError


In [114]:
# Test: angle_difference
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [115]:
def score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: profile (direction -> path count, from swell_window_profile),
    #        buoy_reading (peak_energy in m²/Hz, peak_period in s, peak_direction in degrees, height in ft),
    #        wind (speed in knots, direction in degrees, where it comes FROM),
    #        offshore_direction in degrees (wind direction that blows offshore at this spot),
    #        cutoffs (energy_min, period_min, exposure_min, light_wind_max, offshore_tolerance)
    # Returns: verdict ("worth checking", "not worth it", or "not enough data"),
    #          reasons (list of strings), swell numbers for display (or NONE)
    #
    # if any buoy_reading or wind value is missing:
    #     return "not enough data", ["missing buoy or wind data"], NONE   # EDGE CASE
    #
    # reasons = empty list
    #
    # --- Is there something? ---
    # nearest_direction = buoy peak_direction rounded to the nearest 10 degrees (360 -> 0)
    # exposed = profile[nearest_direction] >= cutoffs.exposure_min
    # enough_energy = buoy peak_energy >= cutoffs.energy_min
    # long_enough = buoy peak_period >= cutoffs.period_min
    # something = exposed and enough_energy and long_enough
    # for each check that failed: add an explanation to reasons
    #     # i.e., "spot is blocked from this swell direction", "low wave energy"
    #
    # --- Is it clean? ---
    # light_wind = wind speed <= cutoffs.light_wind_max
    # offshore = angle_difference(wind direction, offshore_direction) <= cutoffs.offshore_tolerance
    # clean = light_wind or offshore
    # if not clean: add "onshore or cross-shore wind" to reasons
    #
    # if something and clean:
    #     verdict = "worth checking"
    # else:
    #     verdict = "not worth it"
    #
    # return verdict, reasons, (height, peak_period, peak_direction)
    raise NotImplementedError


In [116]:
# Test: score_conditions
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 4. Full run


In [117]:
def run_forecast(pin, bathymetry_file, bounds, buoy_reading, wind, offshore_direction, cutoffs, radius, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: pin (row, col), bathymetry_file (string), bounds (south, north, west, east),
    #        buoy_reading, wind, offshore_direction, cutoffs (as in score_conditions),
    #        radius (int), neighbors (8 or 16)
    # Returns: report (verdict, reasons, swell numbers, swell window profile)
    #
    # depth_grid, cell_size = load_bathymetry(bathymetry_file, bounds)
    #
    # if pin is outside the grid:
    #     raise error "pin is outside the bathymetry bounds"           # EDGE CASE
    #
    # period = buoy_reading.peak_period
    # directions = [0, 10, 20, ..., 350]
    #
    # profile = swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors)
    # verdict, reasons, swell_numbers = score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs)
    #
    # return report containing verdict, reasons, swell_numbers, profile
    raise NotImplementedError


In [118]:
# Test: run_forecast
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)
